# P2P / PnP Editing Experiments
#
# Tests two editing modes on a real image:
# 1. **Cross-attention only** (P2P word swap)
# 2. **Cross-attention + self-attention** (P2P + PnP combined)
#
# Requires: a completed inversion run (or will run one from scratch).

In [ ]:
# 1. Setup
!rm -rf real-image-editing
!git clone https://github.com/beratcelik1/real-image-editing.git
%cd real-image-editing
!pip install -q torch diffusers transformers accelerate safetensors Pillow numpy torchvision scikit-image tqdm matplotlib lpips open-clip-torch

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# 2. HuggingFace login
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"))
    print("Logged in via Colab secret")
except Exception:
    import os
    if os.environ.get("HF_TOKEN"):
        login(token=os.environ["HF_TOKEN"])
        print("Logged in via env var")
    else:
        print("WARNING: No HF token found. Add 'HF_TOKEN' in Colab secrets.")

In [ ]:
# 3. Load model components
from src.utils import (
    load_sd_components, load_image, encode_image,
    decode_latent, get_text_embeddings, get_uncond_embeddings, get_device,
)
from src.inversion import ddim_inversion, null_text_optimization

device = get_device()
print(f"Device: {device}")

print("Loading Stable Diffusion 2.1...")
unet, vae, text_encoder, tokenizer, scheduler = load_sd_components(device)
print("Model loaded!")

In [ ]:
# 4. Download image and define prompts
!wget -q -O data/sample.jpg "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=512"

IMAGE_PATH = "data/sample.jpg"
SOURCE_PROMPT = "a photo of a cat looking at the camera"
TARGET_PROMPT = "a photo of a dog looking at the camera"

NUM_STEPS = 50
CFG_SCALE = 7.5
OPT_STEPS = 10
LR = 1e-2

image = load_image(IMAGE_PATH)
print(f"Source: {SOURCE_PROMPT}")
print(f"Target: {TARGET_PROMPT}")

from IPython.display import display
display(image)

In [ ]:
# 5. Inversion — load from cache if available, otherwise run from scratch
import os
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)
cache_path = "outputs/inversion_cache.pt"

if os.path.exists(cache_path):
    print("Loading inversion cache...")
    cache = torch.load(cache_path, map_location=device, weights_only=True)
    latent_T = cache["latent_T"]
    latents_trajectory = cache["latents_trajectory"]
    null_embeddings = cache["null_embeddings"]
    source_text_emb = cache["text_embeddings"]
    print(f"Loaded cache (prompt: {cache['prompt']})")
else:
    print("No cache found — running inversion from scratch...")
    latent = encode_image(image, vae, device)
    source_text_emb = get_text_embeddings(SOURCE_PROMPT, tokenizer, text_encoder, device)
    uncond_emb = get_uncond_embeddings(tokenizer, text_encoder, device)

    print("DDIM inversion...")
    latent_T, latents_trajectory = ddim_inversion(
        latent, source_text_emb, uncond_emb, unet, scheduler,
        num_steps=NUM_STEPS, cfg_scale=CFG_SCALE,
    )

    print("Null-text optimization...")
    null_embeddings = null_text_optimization(
        latents_trajectory, source_text_emb, uncond_emb, unet, scheduler,
        num_steps=NUM_STEPS, cfg_scale=CFG_SCALE,
        opt_steps=OPT_STEPS, lr=LR,
    )

    torch.save({
        "null_embeddings": null_embeddings,
        "latent_T": latent_T,
        "latents_trajectory": latents_trajectory,
        "text_embeddings": source_text_emb,
        "prompt": SOURCE_PROMPT,
        "num_steps": NUM_STEPS,
        "cfg_scale": CFG_SCALE,
    }, cache_path)
    print(f"Saved cache to {cache_path}")

# Get target text embeddings
target_text_emb = get_text_embeddings(TARGET_PROMPT, tokenizer, text_encoder, device)
print(f"latent_T: {latent_T.shape}, null_embeddings: {len(null_embeddings)}")

In [ ]:
# 6. Editing helper — denoises with attention injection
#
# Batch layout: [src_uncond, src_cond, tgt_uncond, tgt_cond]
# This keeps the [source…, target…] convention the controllers expect:
#   first half  = source (uncond + cond)
#   second half = target (uncond + cond)

from tqdm import tqdm

def p2p_edit(
    latent_T,
    null_embeddings,
    source_text_emb,
    target_text_emb,
    unet,
    scheduler,
    num_steps=50,
    cfg_scale=7.5,
    controller=None,
):
    """Denoise with P2P/PnP attention injection.

    The controller must already be registered on the UNet via
    register_attention_control / register_combined_control.
    """
    scheduler.set_timesteps(num_steps)
    latent = latent_T.clone()

    for i, t in enumerate(tqdm(scheduler.timesteps, desc="Editing")):
        null_emb = null_embeddings[i]

        # [src_uncond, src_cond, tgt_uncond, tgt_cond]
        latent_input = torch.cat([latent] * 4)
        embed_input = torch.cat([
            null_emb, source_text_emb,
            null_emb, target_text_emb,
        ])

        with torch.no_grad():
            noise_pred = unet(
                latent_input, t, encoder_hidden_states=embed_input,
            ).sample

        # CFG for target only (indices 2, 3)
        noise_uncond_tgt = noise_pred[2:3]
        noise_cond_tgt = noise_pred[3:4]
        noise_cfg = noise_uncond_tgt + cfg_scale * (noise_cond_tgt - noise_uncond_tgt)

        latent = scheduler.step(noise_cfg, t, latent).prev_sample

        if controller is not None:
            controller.step()

    return latent

print("p2p_edit() defined")

## Experiment 1: Cross-attention only (P2P word swap)
Replaces "cat" → "dog" by injecting source cross-attention maps into the target for matched tokens.

In [ ]:
# 7. Experiment 1 — cross-attention only
from attention_control.cross_attention import (
    CrossAttentionController,
    compute_token_mapping_from_text,
    register_attention_control,
    unregister_attention_control,
)

# Compute token alignment: "cat" and "dog" differ at one token position;
# all other tokens are mapped so their source attention maps are preserved.
token_mapping = compute_token_mapping_from_text(tokenizer, SOURCE_PROMPT, TARGET_PROMPT)
print(f"Token mapping ({len(token_mapping)} matched tokens): {token_mapping}")

# Show which tokens changed
src_ids = tokenizer.encode(SOURCE_PROMPT)
tgt_ids = tokenizer.encode(TARGET_PROMPT)
src_tokens = tokenizer.convert_ids_to_tokens(src_ids)
tgt_tokens = tokenizer.convert_ids_to_tokens(tgt_ids)
unmapped_src = [i for i in range(len(src_ids)) if i not in token_mapping]
unmapped_tgt = [i for i in range(len(tgt_ids)) if i not in token_mapping.values()]
print(f"Changed source tokens: {[(i, src_tokens[i]) for i in unmapped_src]}")
print(f"Changed target tokens: {[(i, tgt_tokens[i]) for i in unmapped_tgt]}")

cross_ctrl = CrossAttentionController(
    num_steps=NUM_STEPS,
    cross_replace_steps=0.8,
    token_mapping=token_mapping,
)

register_attention_control(unet, cross_ctrl)

print("\nRunning P2P edit (cross-attention only)...")
edited_latent_cross = p2p_edit(
    latent_T, null_embeddings, source_text_emb, target_text_emb,
    unet, scheduler, num_steps=NUM_STEPS, cfg_scale=CFG_SCALE,
    controller=cross_ctrl,
)

unregister_attention_control(unet)

edited_image_cross = decode_latent(edited_latent_cross, vae)
print("Done!")
display(edited_image_cross)

## Experiment 2: Cross-attention + self-attention (P2P + PnP)
Same word swap, but also injects source self-attention maps to preserve spatial layout.

In [ ]:
# 8. Experiment 2 — cross-attention + self-attention (combined)
#
# Two controllers need independent step() calls, so we inline the loop
# rather than using p2p_edit().

from attention_control.self_attention import (
    SelfAttentionController,
    register_combined_control,
    unregister_attention_control as unregister,
)

cross_ctrl_2 = CrossAttentionController(
    num_steps=NUM_STEPS,
    cross_replace_steps=0.8,
    token_mapping=token_mapping,
)

self_ctrl = SelfAttentionController(
    num_steps=NUM_STEPS,
    self_replace_steps=0.5,
)

register_combined_control(unet, cross_ctrl_2, self_ctrl)

scheduler.set_timesteps(NUM_STEPS)
latent = latent_T.clone()

print("Running P2P+PnP edit (cross + self attention)...")
for i, t in enumerate(tqdm(scheduler.timesteps, desc="Editing (P2P+PnP)")):
    null_emb = null_embeddings[i]
    latent_input = torch.cat([latent] * 4)
    embed_input = torch.cat([null_emb, source_text_emb, null_emb, target_text_emb])

    with torch.no_grad():
        noise_pred = unet(latent_input, t, encoder_hidden_states=embed_input).sample

    noise_uncond_tgt = noise_pred[2:3]
    noise_cond_tgt = noise_pred[3:4]
    noise_cfg = noise_uncond_tgt + CFG_SCALE * (noise_cond_tgt - noise_uncond_tgt)
    latent = scheduler.step(noise_cfg, t, latent).prev_sample

    cross_ctrl_2.step()
    self_ctrl.step()

unregister(unet)

edited_image_combined = decode_latent(latent, vae)
print("Done!")
display(edited_image_combined)

## Side-by-side comparison

In [ ]:
# 9. Side-by-side comparison
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

axes[0].imshow(image)
axes[0].set_title(f"Original\n\"{SOURCE_PROMPT}\"", fontsize=11)
axes[0].axis("off")

axes[1].imshow(edited_image_cross)
axes[1].set_title(f"Cross-attn only (P2P)\n\"{TARGET_PROMPT}\"", fontsize=11)
axes[1].axis("off")

axes[2].imshow(edited_image_combined)
axes[2].set_title(f"Cross + self-attn (P2P+PnP)\n\"{TARGET_PROMPT}\"", fontsize=11)
axes[2].axis("off")

plt.tight_layout()
plt.savefig("outputs/edit_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved to outputs/edit_comparison.png")

In [ ]:
# 10. Save individual outputs
edited_image_cross.save("outputs/edit_cross_only.png")
edited_image_combined.save("outputs/edit_cross_self.png")
image.save("outputs/original.png")

print("Saved:")
print("  outputs/original.png")
print("  outputs/edit_cross_only.png")
print("  outputs/edit_cross_self.png")
print("  outputs/edit_comparison.png")

In [ ]:
# 10. Evaluate edit quality with metrics
from src.metrics import compute_ssim, compute_lpips
from skimage.metrics import peak_signal_noise_ratio
import numpy as np

# Evaluate cross-attention edit
psnr_cross = peak_signal_noise_ratio(np.array(image), np.array(edited_image_cross), data_range=255.0)
ssim_cross = compute_ssim(image, edited_image_cross)
lpips_cross = compute_lpips(image, edited_image_cross)

print("Cross-attention Edit (P2P) Metrics:")
print(f"PSNR: {psnr_cross:.2f} dB")
print(f"SSIM: {ssim_cross:.4f}")
print(f"LPIPS: {lpips_cross:.4f}")

# Evaluate combined edit
psnr_combined = peak_signal_noise_ratio(np.array(image), np.array(edited_image_combined), data_range=255.0)
ssim_combined = compute_ssim(image, edited_image_combined)
lpips_combined = compute_lpips(image, edited_image_combined)

print("\nCombined Edit (P2P+PnP) Metrics:")
print(f"PSNR: {psnr_combined:.2f} dB")
print(f"SSIM: {ssim_combined:.4f}")
print(f"LPIPS: {lpips_combined:.4f}")

print("\nNote: Directional similarity metrics (CLIP, DINO) require a target image, which is not available in this prompt-based experiment.")